# HỆ THỐNG GIÁM SÁT VÀ PHÁT HIỆN SỰ MẤT TẬP TRUNG CỦA TÀI XẾ

## Fine tune model mắt khi đeo kính

Chạy GPU, fine-tune C2f cuối và Classify từ eye_refined_v4. MRL có split theo người cho giai đoạn fine-tune này; dữ liệu RGB cũ giữ nguyên các split để kiểm tra hồi quy. Không dùng test để chọn epoch. Chưa chứng minh MRL độc lập với dữ liệu train checkpoint gốc, và audit trước đó đã xem một phần MRL. Các phép đo là phát triển, không phải nghiệm thu video RGB người mới/camera USB/Radxa.

Sampling: RGB rehearsal 55%, MRL kính 30%, MRL không kính 15%, cân bằng lớp. Validation chặn suy giảm RGB/không kính; cần tăng F1 kính ít nhất 3 điểm. Giữ nguyên preprocessing và ngưỡng runtime. Nếu không checkpoint nào đạt, chỉ xuất báo cáo và không thay model. Hiện chưa có video RGB đeo kính của người dùng.


In [ ]:
import sys,subprocess,os,json
from pathlib import Path
from importlib.metadata import version
assert (3,9)<=sys.version_info[:2]<(3,14)
os.environ['YOLO_CONFIG_DIR']='/kaggle/working/.config/ultralytics'
os.environ['MPLCONFIGDIR']='/kaggle/working/.config/matplotlib'
requirements=Path('/kaggle/working/requirements-glasses.txt')
requirements.write_text('# Keep Kaggle\'s CUDA-enabled torch/torchvision; do not replace them with CPU wheels.\nultralytics==8.4.163\nonnx==1.17.0; python_version < "3.13"\nonnx==1.19.1; python_version >= "3.13"\nonnxruntime==1.20.1; python_version < "3.13"\nonnxruntime==1.23.2; python_version >= "3.13"\nnumpy==1.26.4; python_version < "3.13"\nnumpy==2.2.6; python_version >= "3.13"\nPillow==11.1.0\nopencv-python==4.11.0.86\n',encoding='utf-8')
constraints=Path('/kaggle/working/constraints-gpu.txt')
constraints.write_text(''.join(f'{name}=={version(name)}\n' for name in ('torch','torchvision')),encoding='utf-8')
subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--only-binary=:all:','-c',str(constraints),'-r',str(requirements)])


In [ ]:
matches=[path for path in Path('/kaggle/input').rglob('bundle.json') if json.loads(path.read_text()).get('kind')=='driver_monitor_glasses_adaptation']
assert len(matches)==1
bundle_root=matches[0].parent
subprocess.check_call([sys.executable,'-u',str(bundle_root/'kaggle_glasses_training.py'),
                      '--root',str(bundle_root),'--output','/kaggle/working/glasses_model','--epochs','24','--patience','7'])


In [ ]:
report=json.loads(Path('/kaggle/working/glasses_model/evaluation.json').read_text())
print('Status:',report['status'],'selected epoch:',report['selected_epoch'])
if 'test' in report:
    for version_name,groups in report['test'].items():
        for name,metric in groups.items():
            print(version_name,name,'F1:',metric['macro_f1'],'accuracy:',metric['accuracy'])
    from IPython.display import display,Image
    for file in Path('/kaggle/working/glasses_model').glob('*mrl_glasses_matrix.png'):
        display(Image(filename=str(file)))
print('Download /kaggle/working/glasses_result.zip')
